## 랭그래프 실습

1. 질문 : 체력이 안좋고 살이 계속 찌는데 어떤 운동을 할까?

2. 추출 에이전트 : 사용자의 질문에서 증상을 추출

3. 후보 에이전트 : 증상을 해결할 수 있는 운동 리스트 추출

4. 답변 생성 에이전트 : 증상과 운동리스트를 개조식으로 출력하도록 답변 생성

In [ ]:
from langchain_community.llms import Ollama  # Ollama LLM 사용
from langchain_core.prompts import PromptTemplate  # 프롬프트 템플릿
from langgraph.graph import StateGraph, END  # LangGraph 상태 머신
from typing import TypedDict  # 타입 정의용

# 1. 상태 정의
class AgentState(TypedDict):  # 상태 타입 정의
    query: str  # 사용자 질의
    symptoms: str  # 추출된 증상
    exercise_candidates: str  # 해결 운동 후보
    result: str  # 최종 응답

# 2. LLM 초기화
llm = Ollama(model="exaone3.5:2.4b")  # Ollama 모델 로딩

In [ ]:
# 3. 에이전트 정의

# 사용자에서 증상을 추출하라고 추출에이전트에 넣을 질의
extractor_prompt = PromptTemplate.from_template("""
                                                사용자의 질문에서 증상에 해당하는 단어 또는 구를 추출.  
                                                결과는 쉼표로 구분된 문자열로 출력.  
                                                질문: {query}
                                                """)  # 증상 추출 프롬프트

# 에이전트 모듈함수 (인자에 반드시 Agent State를 입력해야함. (2 in 1 구조에서 예외가 있기는 하다.))
def extractor_agent(state: AgentState):  # 증상 추출 함수
    chain = extractor_prompt | llm  # 프롬프트 체인 (LCEL)
    symptoms = chain.invoke({"query": state["query"]})  # LLM 실행

    # 사용자 : 나는 체력이 안좋고, 살이 계속 찐다.
    # symptoms : "체력 저하, 체중 증가"
    return {**state, "symptoms": symptoms.strip()}  # 상태에 추가, AgentState class의 state인스턴스에 symptoms 변수값을 수정하는 부분

# 추출에이전트에서 뽑은 symptoms를 가지고 해결할 수 있는 운동리스트를 추론하는 의사에이전트.
matcher_prompt = PromptTemplate.from_template("""
                                                다음 증상 목록을 바탕으로 가장 해결 가능성 높은 운동 이름 3개를 쉼표로 추정.
                                                증상: {symptoms}
                                                """)  # 질병 후보 추정 프롬프트

# 추출에이전트에서 얻은 질병을 가지고 해결가능한 운동을 추론하는 파이프함수
def matcher_agent(state: AgentState):  # 질병 후보 추정
    chain = matcher_prompt | llm # LCEL
    candidates = chain.invoke({"symptoms": state["symptoms"]}) # state["symptoms"]: AgentState class의 인스턴스(state)에서
                                                               # symptoms 값을 가져와 질의로 입력
    return {**state, "exercise_candidates": candidates.strip()}

# 사용자의 증상과 질병 후보를 받아서 최종답변을 생성하는 에이전트
answer_prompt = PromptTemplate.from_template("""
                                            사용자의 증상: {symptoms}

                                            예측된 운동 후보: {exercise_candidates}

                                            위 내용을 바탕으로 사용자에게 알기 쉽게 개조식으로 설명.
                                            """)  # 최종 응답 생성 프롬프트

# 최종답변 함수
def answer_agent(state: AgentState):  # 응답 생성 에이전트
    chain = answer_prompt | llm  # 프롬프트와 LLM을 연결하여 실행 체인 구성
    answer = chain.invoke({
        "symptoms": state["symptoms"], # Agent State의 인스턴스 sgtate의 symptoms 값 가져옴
        "exercise_candidates": state["exercise_candidates"] # Agent State의 인스턴스 sgtate의 exercise_candidates 값 가져옴
    })
    return {**state, "result": answer.strip()}

# 4. LangGraph 정의
from langgraph.graph import StateGraph  # LangGraph 구성 요소

graph = StateGraph(AgentState)  # 그래프 정의
graph.add_node("extractor", extractor_agent)  # 노드 추가
graph.add_node("matcher", matcher_agent)
graph.add_node("answer", answer_agent)

graph.set_entry_point("extractor")  # 시작 노드 설정
graph.add_edge("extractor", "matcher")  # 노드 간 연결 정의
graph.add_edge("matcher", "answer")
graph.add_edge("answer", END)  # 종료 노드 설정

app = graph.compile()  # 그래프 컴파일

In [ ]:
# 5. 실행 예시
query = "체력이 안좋고, 살이 계속 찐다"  # 사용자 질문
result = app.invoke({"query": query})  # 실행

print("============================== 최종 응답:")
print(result["result"])  # 결과 출력